In [1]:
import pandas as pd
df = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")

In [2]:
df.to_csv("submission.csv",index = False)

MILESTONE 1 STARTS 

In [8]:
import pandas as pd
import numpy as np

# Load the dataset (Kaggle environment path)
df = pd.read_csv('/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv')

# Create a filtered dataframe that only contains rows where a defect actually exists.
# We will use this 'defective_rows' variable in all the following cells.
df['has_defect'] = df['mask_rle'].notna() & (df['mask_rle'].str.strip() != '')
defective_rows = df[df['has_defect']]

print(f"Total rows: {len(df)}")
print(f"Rows with actual defects: {len(defective_rows)}")

Total rows: 45572
Rows with actual defects: 7201


QN 1 

In [9]:
# Find the count of the most frequent defect class
class_counts = defective_rows['defect_type'].value_counts()
most_frequent_count = class_counts.max()

# Find the number of completely clean images (Total unique images - defective unique images)
total_unique_images = df['image_id'].nunique()
defective_unique_images = defective_rows['image_id'].nunique()
clean_images_count = total_unique_images - defective_unique_images

# Calculate the final answer
q1_answer = most_frequent_count + clean_images_count

print(f"Most frequent class count: {most_frequent_count}")
print(f"Clean images count: {clean_images_count}")
print(f"Question 1 Answer: {q1_answer}")

Most frequent class count: 4100
Clean images count: 5516
Question 1 Answer: 9616


Q2

In [10]:
# Count how many unique defect types each image has
defects_per_image = defective_rows.groupby('image_id')['defect_type'].nunique()

# Filter for images that have strictly more than 1 defect class
# We save this as a new variable because Question 3 will need it!
multi_label_images_series = defects_per_image[defects_per_image > 1]

q2_answer = len(multi_label_images_series)
print(f"Question 2 Answer (Images with >1 defect): {q2_answer}")

Question 2 Answer (Images with >1 defect): 1198


Q3

In [11]:
def rle_decode(mask_rle, shape=(128, 800)):
    """Decodes Kaggle RLE strings into a 2D binary image mask."""
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if pd.isna(mask_rle) or not isinstance(mask_rle, str):
        return img.reshape(shape)
    
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1  # Adjusting for 1-based indexing
    ends = starts + lengths
    
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
        
    return img.reshape(shape, order='F') 

# Get the exact image IDs from our Question 2 results
multi_label_image_ids = multi_label_images_series.index
total_overlapping_pixels = 0

# Test every image that has multiple labels
for img_id in multi_label_image_ids:
    # Filter our global defective_rows for just this image
    img_data = defective_rows[defective_rows['image_id'] == img_id]
    
    # Create an empty canvas
    combined_mask = np.zeros((128, 800), dtype=np.uint8)
    
    for rle in img_data['mask_rle']:
        # Add each defect mask to the canvas
        combined_mask += rle_decode(rle)
        
    # Count pixels where two or more masks overlap
    total_overlapping_pixels += np.sum(combined_mask >= 2)

print(f"Question 3 Answer (Total Overlapping Pixels): {total_overlapping_pixels}")

Question 3 Answer (Total Overlapping Pixels): 0


In [12]:
# QUESTION 4: The Boundary Edge Case 


# The absolute maximum number of pixels allowed on our canvas
max_allowed_pixels = 128 * 800   # 102,400

malformed_row_count = 0
global_max_index = 0

# We only need to check the rows that actually have RLE strings
for rle_string in defective_rows['mask_rle']:
    
    # Split the string into numbers
    tokens = [int(x) for x in rle_string.split()]
    
    # Grab the starts and lengths
    # Starts are at index 0, 2, 4... Lengths are at index 1, 3, 5...
    starts = tokens[0::2]
    lengths = tokens[1::2]
    
    # For every run in this string, calculate where it ends
    row_is_malformed = False
    
    for start, length in zip(starts, lengths):
        # Calculate the 1-based end index: start + length - 1
        end_index = start + length - 1
        
        # Keep track of the absolute highest index we ever see
        if end_index > global_max_index:
            global_max_index = end_index
            
        # If the end index is bigger than 102,400, this row is broken!
        if end_index > max_allowed_pixels:
            row_is_malformed = True
            
    # If any run in this row went out of bounds, we count the row
    if row_is_malformed:
        malformed_row_count += 1

# Print the final result in the exact format the form asks for
print(f"Question 4 Answer: {malformed_row_count} rows, max index {global_max_index}")

Question 4 Answer: 0 rows, max index 102400


In [14]:

# QUESTION 5: Geometric Distortion & Anisotropic Aspect Ratio


# Original Image Dimensions
original_height = 128
original_width = 800

# Target Image Dimensions (The naive square resize)
target_height = 256
target_width = 256

# 1. Compute Horizontal Scale Factor (s_x)
# How much are we scaling the width?
s_x = target_width / original_width 

# 2. Compute Vertical Scale Factor (s_y)
# How much are we scaling the height?
s_y = target_height / original_height

# 3. Compute the Distortion Ratio (s_y / s_x)
# How much worse is the stretching on the Y axis compared to the X axis?
distortion_ratio = s_y / s_x

# Print the results matching the required form format: `s_x, s_y, ratio`
print(f"Question 5 Answer: {s_x}, {s_y}, {distortion_ratio}")

Question 5 Answer: 0.32, 2.0, 6.25


In [15]:
# 
# QUESTION 6: Photometric Invisibility (Cohen's d)
# 
import cv2 # We need OpenCV to load the actual image files now
import os

# Create a dictionary to hold the pixel values for each defect class
class_stats = {
    'defect_a': {'d_pixels': [], 'bg_pixels': []},
    'defect_b': {'d_pixels': [], 'bg_pixels': []},
    'defect_c': {'d_pixels': [], 'bg_pixels': []},
    'defect_d': {'d_pixels': [], 'bg_pixels': []}
}

image_dir = '/kaggle/input/competitions/industrial-surface-defect-detection/public/train'

# To save time, we will only sample the first 50 images of each defect type

for defect_type in ['defect_a', 'defect_b', 'defect_c', 'defect_d']:
    
    # Get rows for just this defect type
    specific_defects = defective_rows[defective_rows['defect_type'] == defect_type].head(50)
    
    for index, row in specific_defects.iterrows():
        img_id = row['image_id']
        img_path = os.path.join(image_dir, img_id)
        
        # We must add '.jpg' to the image ID to load the file
        if not img_path.endswith('.jpg'):
            img_path += '.jpg'
            
        # Load the image in grayscale (0 means grayscale)
        img = cv2.imread(img_path, 0)
        
        # If the image failed to load, skip to the next one
        if img is None:
            continue
            
        # Decode the mask using our function from Cell 4
        mask = rle_decode(row['mask_rle'])
        
        # Separate the pixels using the mask
        # mask == 1 means it is a defect pixel
        # mask == 0 means it is a clean background pixel
        defect_pixels = img[mask == 1]
        background_pixels = img[mask == 0]
        
        # Store these pixels in our dictionary
        class_stats[defect_type]['d_pixels'].extend(defect_pixels)
        class_stats[defect_type]['bg_pixels'].extend(background_pixels)

# Now calculate Cohen's d for each class
for defect_type, data in class_stats.items():
    d_pixels = np.array(data['d_pixels'])
    bg_pixels = np.array(data['bg_pixels'])
    
    # Calculate the Means (averages)
    mean_defect = np.mean(d_pixels)
    mean_bg = np.mean(bg_pixels)
    
    # Calculate the Standard Deviations (spreads)
    std_defect = np.std(d_pixels)
    std_bg = np.std(bg_pixels)
    
    # Number of samples
    n_defect = len(d_pixels)
    n_bg = len(bg_pixels)
    
    # Calculate the Pooled Standard Deviation formula
    pooled_std = np.sqrt(((n_defect - 1) * std_defect**2 + (n_bg - 1) * std_bg**2) / (n_defect + n_bg - 2))
    
    # Calculate Cohen's d
    cohens_d = abs(mean_defect - mean_bg) / pooled_std
    
    print(f"{defect_type} Cohen's d: {cohens_d:.4f}")
    
    if cohens_d <= 0.02:
        print(f"--> {defect_type} is practically invisible to thresholds! It requires TEXTURE features.")

defect_a Cohen's d: 0.3558
defect_b Cohen's d: 0.9818
defect_c Cohen's d: 0.2207
defect_d Cohen's d: 0.0729


In [17]:

# QUESTION 7: Metric Behavior (Dice Step Function)


def calculate_dice(pred_pixels, true_pixels):
    """Calculates the Dice score based on standard Kaggle/Industry conventions."""
    # Convention Case 1: Both prediction and ground truth are empty.
    # The model correctly recognized a clean image. Full credit.
    if pred_pixels == 0 and true_pixels == 0:
        return 1.0
        
    # Standard Formula: (2 * Intersection) / (Prediction + Ground Truth)
    # If the ground truth is 0, the intersection (overlap) is always 0.
    intersection = 0 
    
    dice_score = (2.0 * intersection) / (pred_pixels + true_pixels)
    return dice_score

# Ground truth: The image is completely clean
ground_truth_pixels = 0

# Model 1 predicts no defects (0 pixels)
model_1_pixels = 0
dice_model_1 = calculate_dice(model_1_pixels, ground_truth_pixels)

# Model 2 hallucinates 2 stray noise pixels
model_2_pixels = 2
dice_model_2 = calculate_dice(model_2_pixels, ground_truth_pixels)

# Print the final result in the exact format the form asks for
print(f"Question 7 Answer: {dice_model_1:.1f}, {dice_model_2:.1f}")

Question 7 Answer: 1.0, 0.0


In [18]:
#  question 8 - toy batch metric calculation
# testing the official kaggle metric formula by hand

# foreground dice is only for the 10 images with real defects
# 6 images got 0.80 and 4 images got 0.40
sum_dice = (6 * 0.80) + (4 * 0.40)
foreground_dice = sum_dice / 10

# false positive score is only for the 40 clean images
# canvas is 128x800 = 102400. 4 images had 1024 wrong pixels predicted.
fp_fraction = 1024 / 102400
sum_fp = (36 * 0.0) + (4 * fp_fraction)
mean_fp = sum_fp / 40
false_positive_score = 1.0 - mean_fp

# empty accuracy: 36 out of 40 clean images were perfectly clean
empty_accuracy = 36 / 40

# final score using the assignment weights (70%, 20%, 10%)
final_score = (0.70 * foreground_dice) + (0.20 * false_positive_score) + (0.10 * empty_accuracy)

# printing exactly what the google form asked for (a, b, c, d)
print(f"Question 8 Answer: {foreground_dice:.4f}, {false_positive_score:.4f}, {empty_accuracy:.4f}, {final_score:.4f}")

Question 8 Answer: 0.6400, 0.9990, 0.9000, 0.7378


In [19]:
#question 9 - all-empty baseline
# testing what happens if a dumb model just predicts blank for every single image

# 1. naive dice score
# it gets 1.0 on clean images and 0.0 on defect images.
# so the score is literally just the fraction of empty rows in our dataset.
total_rows = len(df) # reusing df from cell 1
empty_rows = total_rows - len(defective_rows) # reusing defective_rows from cell 1
score_naive = empty_rows / total_rows

# 2. official kaggle score for the same blank model
# it finds 0 defects -> foreground dice = 0
# it predicts 0 false pixels -> false positive score = 1.0
# it gets all empty images right -> empty accuracy = 1.0
score_official = (0.70 * 0.0) + (0.20 * 1.0) + (0.10 * 1.0)

# the naive score looks artificially high, but the official score blocks the cheating
print(f"Question 9 Answer: {score_naive:.4f}, {score_official:.4f}")

Question 9 Answer: 0.8420, 0.3000


In [20]:
# question 10 - morphological noise filtering (recall ceilings)
# checking how big the actual defects are so we don't accidentally delete real ones 
# when we write our noise filter later.



# we need lists to store the area (size in pixels) of every single defect "blob"
areas_defect_a = []
areas_defect_b = []

# we only care about defect_a and defect_b for this question
target_classes = ['defect_a', 'defect_b']

# filter our main dataframe for just these two classes
target_rows = defective_rows[defective_rows['defect_type'].isin(target_classes)]

for index, row in target_rows.iterrows():
    defect_type = row['defect_type']
    
    # decode the text string into a 2D image mask (using the function from cell 4)
    mask = rle_decode(row['mask_rle'])
    
    # use OpenCV to find all the separate "blobs" (connected components) in this mask
    # connectivity=8 means diagonal pixels count as being connected
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    
    # loop through all the blobs found in this mask
    # we start at i=1 because i=0 is the background (the black area)
    for i in range(1, num_labels):
        # cv2.CC_STAT_AREA gives us the exact number of pixels in this blob
        blob_area = stats[i, cv2.CC_STAT_AREA]
        
        # save the blob size to the correct list
        if defect_type == 'defect_a':
            areas_defect_a.append(blob_area)
        elif defect_type == 'defect_b':
            areas_defect_b.append(blob_area)

# convert lists to numpy arrays to do math easily
areas_a = np.array(areas_defect_a)
areas_b = np.array(areas_defect_b)

# --- Part 1: Smallest defect_b component ---
# find the absolute smallest real defect_b blob in the whole dataset
min_area_b = np.min(areas_b)

# --- Part 2: Pixel loss for defect_b with N = 50 ---
# if we delete every blob smaller than 50 pixels, how many pixels do we lose?
pixels_lost_b_50 = np.sum(areas_b[areas_b < 50])
total_pixels_b = np.sum(areas_b)
lost_pct_b_50 = (pixels_lost_b_50 / total_pixels_b) * 100

# --- Part 3: Pixel loss for defect_a with N = 200 ---
# if we delete every blob smaller than 200 pixels, how many pixels do we lose?
pixels_lost_a_200 = np.sum(areas_a[areas_a < 200])
total_pixels_a = np.sum(areas_a)
lost_pct_a_200 = (pixels_lost_a_200 / total_pixels_a) * 100

# print the exact format the form asked for (value, percentage, percentage)
print(f"Question 10 Answer: {min_area_b}, {lost_pct_b_50:.4f}, {lost_pct_a_200:.4f}")

Question 10 Answer: 93, 0.0000, 15.2171
